# Can we trust the model's confidence score?

Every labeled block comes with a confidence — the model saying how sure it is.
If that number were trustworthy, wrong labels would come with low confidence.

So: **show the wrong labels, with the confidence the model attached to them.**
Path A, block level, all 10 documents.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    containment, extract_gold, resolve_old_gt_path, tokenize,
)

MODELS, EXTRACTOR, SAMPLES = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b'], 'pdfplumber', range(1, 11)
pd.set_option('display.max_colwidth', 66)
INK, WRONGBG = '#111111', '#fdecea'


def block_truth(n):
    """True label per block, from the annotation — same rules as notebook 7
    (tiny blocks and blocks spanning two items are left out)."""
    gold = extract_gold(resolve_old_gt_path(n))
    blocks = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'sample{n}.json')
                        .read_text(encoding='utf-8'))
    out = []
    for b in blocks:
        bt = tokenize(b['text'])
        if len(bt) < 3:
            out.append(None); continue
        best, lab, bi = 0.0, None, None
        for gi, (gt, gl) in enumerate(gold):
            c = containment(bt, tokenize(gt))
            if c > best:
                best, lab, bi = c, gl, gi
        if best < 0.75:
            out.append(None); continue
        spans = any(gi != bi and len(tokenize(gt)) >= 2
                    and containment(tokenize(gt), bt) >= 0.9
                    for gi, (gt, gl) in enumerate(gold))
        out.append(None if spans else lab)
    return out, blocks


# One row per judgeable block, per model.
rows = []
for n in SAMPLES:
    truth, blocks = block_truth(n)
    for m in MODELS:
        bl = json.loads(P.labeled_path(P.make_tag(m, EXTRACTOR), n)
                        .read_text(encoding='utf-8'))
        for t, b in zip(truth, bl):
            if t is None:
                continue
            rows.append({'model': m, 'sample': n, 'true label': t,
                         'model said': b.get('label'),
                         'confidence': float(b.get('confidence', 1.0)),
                         'correct': b.get('label') == t,
                         'text': b['text']})
df = pd.DataFrame(rows)
print(f'{len(df) // len(MODELS)} judgeable blocks per model')


674 judgeable blocks per model


## Step 1 — Wrong labels, sorted by how confident the model was

For each model: its ten most confident **mistakes**. If confidence were
trustworthy, this table should show low numbers. It does not.


In [2]:
for m in MODELS:
    bad = (df[(df['model'] == m) & (~df['correct'])]
           .sort_values('confidence', ascending=False)
           [['confidence', 'true label', 'model said', 'text']]
           .head(10).reset_index(drop=True))
    display(bad.style
            .set_caption(f'{m} — most confident mistakes')
            .set_table_styles([{'selector': 'caption',
                                'props': [('caption-side', 'top'),
                                          ('font-weight', '700'),
                                          ('font-size', '110%'),
                                          ('text-align', 'left'),
                                          ('color', INK)]}])
            .background_gradient(cmap='Reds', subset=['confidence'],
                                 vmin=0.5, vmax=1.0)
            .format({'confidence': '{:.2f}'}))
    print()


,confidence,true label,model said,text
0,1.00,answer.text,question.text,It is understood that the NSF Engineering Directorate requires data to be made available for a minimum
1,1.00,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
2,1.00,answer.text,question.text,"views of the National Science Foundation.” In keeping with standard ethical practices, it is expected that"
3,1.00,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
4,1.00,answer.text,section.description,them to Monte Carlo simulations. CASM organizes all the files associated with VASP calculations
5,1.00,answer.text,section.description,effective Hamiltonian construction and the large number of Monte Carlo simulations that are
6,1.00,answer.text,section.description,The supporting information accompanying publications will include detailed
7,1.00,answer.text,section.description,"performed by different users are in a format that is readily accessible, either by CASM or directly by"
8,1.00,answer.text,section.title,those of the author(s) and do not necessarily reflect the views of the National Science Foundation.”
9,1.00,answer.text,section.title,approved by a curator before they are published.


,confidence,true label,model said,text
0,1.00,question.text,section.title,"Data Types and Sources. A brief, high-level description of the data to be generated or used through"
1,1.00,question.text,section.title,"Content and Format. A statement of plans for data and metadata content and format including, where"
2,1.00,question.text,section.title,Data Sharing and Data Preservation. A description of the plans for data sharing and preservation.
3,1.00,question.text,section.title,Rationale. A discussion of the rationale or justification for the proposed data management plan
4,0.95,answer.text,section.title,Produced Data: Datasets will be produced in Sections 2.1-2.3 of the proposal. These datasets will be
5,0.95,answer.text,question.text,"The project will result in new sets of data, code, and tools. These outputs include: (a) source code to"
6,0.95,answer.text,question.text,"quality control, collate, and merge datasets, (b) open source models and budgets in R and/or python, and"
7,0.95,answer.text,question.text,(c) sharable instructional materials (Scientist Spotlight activities and workshop materials).
8,0.95,answer.text,section.title,Input data: Available data will be systematically reviewed and downloaded. These data are likely
9,0.95,answer.text,question.text,documented for reproducibility purposes. The metadata for the codes will include: (1) description of the


,confidence,true label,model said,text
0,1.00,answer.text,section.description,The following data will be created as a result of this project:
1,1.00,question.text,section.title,"Data Types and Sources. A brief, high-level description of the data to be generated or used through"
2,1.00,question.text,section.title,"Content and Format. A statement of plans for data and metadata content and format including, where"
3,1.00,question.text,section.title,Data Sharing and Data Preservation. A description of the plans for data sharing and preservation.
4,1.00,question.text,section.title,Rationale. A discussion of the rationale or justification for the proposed data management plan
5,0.90,question.text,section.description,"the selection of appropriate standards. (Existing, accepted community standards should be used where"
6,0.90,question.text,section.description,develop or generalize standards.)
7,0.90,question.text,section.description,"possible. Where community standards are missing or inadequate, the DMP could propose alternate"
8,0.90,question.text,section.description,"strategies that facilitate data sharing, and should advise the sponsoring program of any need to"
9,0.90,answer.text,section.title,Input data: Available data will be systematically reviewed and downloaded. These data are likely


Example of what this means: llama3.1 labels `B. Scientific data that will be
preserved…` as a section heading **with confidence 1.00** — complete certainty,
completely wrong. The model is not unsure when it errs; it is sure and wrong.


## Step 2 — How much of each model's wrongness is high-confidence?

One number per model: of all its wrong labels, how many carried a confidence of
0.9 or more? And for comparison, the average confidence when right vs when wrong.


In [3]:
rows = []
for m in MODELS:
    d = df[df['model'] == m]
    ok, bad = d[d['correct']], d[~d['correct']]
    rows.append({'model': m,
                 'wrong labels': len(bad),
                 'wrong with confidence >= 0.9': (bad['confidence'] >= 0.9).mean(),
                 'avg confidence when RIGHT': ok['confidence'].mean(),
                 'avg confidence when WRONG': bad['confidence'].mean()})
t = pd.DataFrame(rows).set_index('model')
display(t.style
        .background_gradient(cmap='Reds',
                             subset=['wrong with confidence >= 0.9'])
        .format({'wrong with confidence >= 0.9': '{:.0%}',
                 'avg confidence when RIGHT': '{:.2f}',
                 'avg confidence when WRONG': '{:.2f}'}))


,wrong labels,wrong with confidence >= 0.9,avg confidence when RIGHT,avg confidence when WRONG
model,,,,
llama3.1:8b,261,60%,0.87,0.88
gemma4:e4b,100,83%,0.95,0.90
llama3.3:70b,115,40%,0.84,0.82


## Conclusion

**The confidence score cannot be trusted to find errors.**

- **llama3.1:8b** is *more* confident when wrong (0.88) than when right (0.87).
  Its signature mistake — lettered questions labeled as headings — mostly comes
  with confidence 0.95–1.00.
- **llama3.3:70b** looks similar: right and wrong labels carry nearly the same
  confidence, so the number cannot separate them.
- **gemma4:e4b** is the partial exception — its wrong labels are noticeably less
  confident (0.90 vs 0.95), so *very* low values are worth a second look. But even
  there, most mistakes still carry 0.9 or above.

Practical rule: **do not use confidence to filter or auto-accept labels.** A high
number does not mean a right label — the most confident mistakes above are at 1.00.
